# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:36<00:00, 12.02s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

'Title: Open-Box Onn 24" 1080p FreeSync Gaming Monitor + free shipping\nDetails: Apply promo code "TAKE8OFFSALE" to drop this open-box Onn 24" gaming monitor to $52, down from a regular price of $159. It\'s the best deal we\'ve seen for this Onn monitor. The panel runs at 165Hz with a 1ms response time and includes both a DisplayPort and HDMI cable. Coupon ends today. Shop Now at eBay\nFeatures: 24" FHD 1080p display 165Hz refresh rate 1ms response time AMD FreeSync support VESA mount compatible Includes 6ft DisplayPort & HDMI cables\nURL: https://www.dealnews.com/Open-Box-Onn-24-1080-p-Free-Sync-Gaming-Monitor-free-shipping/21836627.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Open-box Cameras, Camcorders & Drone Deals at Best Buy: Up to 50% off + free shipping
Details: In Best Buy's Outlet section you'll find deals on a large selection of open-box cameras, drones, accessories, and more. Stock on select models may be limited. Shop Now at Best Buy
Features: 
URL: https://www.dealnews.com/Open-box-Cameras-Camcorders-Drone-Deals-at-Best-Buy-Up-to-50-off

In [8]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description="The Apple TV 4K (3rd generation) is a compact 4K streaming media player powered by the A15 Bionic chip, offering hardware-accelerated Dolby Vision and HDR10+ playback. This 64GB Wi‑Fi model supports Wi‑Fi 6 for faster streaming and includes the Siri Remote with a touch-enabled clickpad for navigation. It's suited for users who want a powerful, up-to-date streamer with ample local storage for apps and games.", price=102.0, url='https://www.dealnews.com/Open-box-3-rd-Gen-Apple-TV-4-K-64-GB-Wi-Fi-Streaming-Media-Player-free-shipping/21836626.html?iref=rss-c142'), Deal(product_description='The Apple MagSafe Wireless Charger (1st gen, model MGD74LL/A) is a compact magnetic wireless charging puck that provides aligned, efficient Qi-based charging for MagSafe‑compatible iPhones and accessories. It includes a 1‑meter USB‑C cable and supports fast wireless charging up to 25W when paired with an appropriate USB‑C power adapter.', price=17.0, url='ht

In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


The Apple TV 4K (3rd generation) is a compact 4K streaming media player powered by the A15 Bionic chip, offering hardware-accelerated Dolby Vision and HDR10+ playback. This 64GB Wi‑Fi model supports Wi‑Fi 6 for faster streaming and includes the Siri Remote with a touch-enabled clickpad for navigation. It's suited for users who want a powerful, up-to-date streamer with ample local storage for apps and games.
102.0
https://www.dealnews.com/Open-box-3-rd-Gen-Apple-TV-4-K-64-GB-Wi-Fi-Streaming-Media-Player-free-shipping/21836626.html?iref=rss-c142

The Apple MagSafe Wireless Charger (1st gen, model MGD74LL/A) is a compact magnetic wireless charging puck that provides aligned, efficient Qi-based charging for MagSafe‑compatible iPhones and accessories. It includes a 1‑meter USB‑C cable and supports fast wireless charging up to 25W when paired with an appropriate USB‑C power adapter.
17.0
https://www.dealnews.com/Open-Box-1-st-Gen-Apple-Mag-Safe-Wireless-Charger-free-shipping/21836622.html?ir

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

DealSelection(deals=[Deal(product_description='The Vizio VQD50R-10 is a 50-inch 4K QLED smart TV that delivers 3840 x 2160 resolution with HDR support including Dolby Vision, HDR10+, HDR10, and HLG. It runs Vizio’s smart TV platform with compatibility for Google Home, Alexa, and Apple Home, and includes three HDMI inputs for connecting consoles and streaming devices. The set targets viewers who want a bright, color-rich panel and modern smart home integration in a mid-priced 50" form factor.', price=238.0, url='https://www.dealnews.com/products/Vizio/Vizio-VQD50-R-10-50-4-K-HDR-QLED-UHD-Smart-TV/498928.html?iref=rss-f1912'), Deal(product_description='The Onn 100012587 is a 65-inch 4K HDR Roku TV offering 3840 x 2160 resolution and HDR10 support, built on the Roku smart-TV platform for easy access to streaming apps. It supports integrations with Apple Home, Amazon Alexa, and Google Home, and includes three HDMI ports for multiple sources. This model provides a large-screen, feature-rich

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [33]:
load_dotenv(override=True)

True

In [34]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [35]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [36]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [37]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [38]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [39]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
18:46:20 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
INFO:LiteLLM:
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
18:46:22 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
